# 07. ViHOS Span Explanation: Rule Baseline + Transformer Token Classifier

Notebook này xây module phát hiện đoạn độc hại trong bình luận.

Mục tiêu:
- tạo rule baseline để có mốc so sánh và fallback cho app;
- train token classification model từ ViHOS để học span theo ngữ cảnh;
- đánh giá rule vs model trên dev/test;
- xuất artifact cho app demo.

## 1. Thư viện

In [1]:
# Nếu thiếu package, chạy một lần trong terminal:
# pip install -U pandas numpy scikit-learn tqdm torch transformers

import ast
import json
import math
import os
import random
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    get_linear_schedule_with_warmup,
)

## 2. Cấu hình

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for p in candidates:
        has_data = (p / "data" / "processed" / "test_processed.csv").exists()
        has_vihos = (p / "data" / "vihos" / "repo" / "data").exists()
        has_outputs = (p / "outputs").exists() or (p / "notebooks").exists()
        if has_data and has_vihos and has_outputs:
            return p.resolve()
    raise FileNotFoundError(
        "Khong tim thay project root. Hay chay notebook trong thu muc notebooks/ "
        "hoac root project co data/processed va data/vihos/repo/data."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
RESULTS_DIR = OUTPUT_DIR / "results"
RESOURCES_DIR = OUTPUT_DIR / "resources"
MODELS_DIR = OUTPUT_DIR / "models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESOURCES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

CONTRACT_PATH = RESOURCES_DIR / "extension_data_contract.json"
TRAIN_STD_PATH = RESOURCES_DIR / "vihos_span_train_standard.csv"
DEV_STD_PATH = RESOURCES_DIR / "vihos_span_dev_standard.csv"
TEST_STD_PATH = RESOURCES_DIR / "vihos_span_test_standard.csv"
PHRASE_PATH = RESOURCES_DIR / "toxic_phrases_candidates_train.csv"

MODEL_NAME = "xlm-roberta-base"
TOKEN_MODEL_DIR = MODELS_DIR / "vihos_xlmr_token_classifier"

RUN_TOKEN_MODEL = True
MAX_LENGTH = 192
BATCH_SIZE = 8
EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
PATIENCE = 2
GRAD_CLIP = 1.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

PROJECT_ROOT: D:\Thư bae ngốc ngếch
DEVICE: cpu


## 3. Đọc dữ liệu từ Note 6

In [3]:
required_files = {
    "contract": CONTRACT_PATH,
    "vihos_train": TRAIN_STD_PATH,
    "vihos_dev": DEV_STD_PATH,
    "vihos_test": TEST_STD_PATH,
    "toxic_phrases": PHRASE_PATH,
}

missing = [name for name, path in required_files.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Thieu output tu Note 6: " + ", ".join(missing) +
        "\nHay chay 06_extension_data_contract_and_vihos_readiness_local.ipynb truoc."
    )

with open(CONTRACT_PATH, "r", encoding="utf-8") as f:
    contract = json.load(f)

train_df = pd.read_csv(TRAIN_STD_PATH).fillna("")
dev_df = pd.read_csv(DEV_STD_PATH).fillna("")
test_df = pd.read_csv(TEST_STD_PATH).fillna("")
phrases_df = pd.read_csv(PHRASE_PATH).fillna("")

for name, df in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    if df.empty:
        raise ValueError(f"{name}_df rong. Kiem tra lai Note 6.")
    if "content" not in df.columns:
        raise ValueError(f"{name}_df thieu cot content.")

if phrases_df.empty:
    raise ValueError("toxic_phrases_candidates_train.csv rong.")

print("train:", train_df.shape)
print("dev:", dev_df.shape)
print("test:", test_df.shape)
print("phrases:", phrases_df.shape)

train: (8844, 13)
dev: (1106, 13)
test: (1106, 13)
phrases: (4426, 5)


## 4. Chuẩn hóa span ground truth

In [4]:
def safe_literal(x):
    if isinstance(x, (list, tuple, dict)):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return []
    try:
        return ast.literal_eval(s)
    except Exception:
        return []


def flatten_ints(obj):
    out = []
    if isinstance(obj, (list, tuple, set)):
        for item in obj:
            out.extend(flatten_ints(item))
    else:
        try:
            out.append(int(obj))
        except Exception:
            pass
    return out


def parse_index_spans(row):
    text = str(row.get("content", ""))
    n = len(text)

    if "index_spans" in row:
        idx = sorted(set(i for i in flatten_ints(safe_literal(row["index_spans"])) if 0 <= i < n))
        if idx:
            return idx

    if "span_ranges" in row:
        ranges = safe_literal(row["span_ranges"])
        idx = []
        if isinstance(ranges, (list, tuple)):
            for r in ranges:
                if isinstance(r, (list, tuple)) and len(r) >= 2:
                    try:
                        start, end = int(r[0]), int(r[1])
                        start = max(0, start)
                        end = min(n, end)
                        if end > start:
                            idx.extend(range(start, end))
                    except Exception:
                        pass
        idx = sorted(set(idx))
        if idx:
            return idx

    return []


def indices_to_ranges(indices):
    indices = sorted(set(int(i) for i in indices))
    if not indices:
        return []
    ranges = []
    start = prev = indices[0]
    for i in indices[1:]:
        if i == prev + 1:
            prev = i
        else:
            ranges.append((start, prev + 1))
            start = prev = i
    ranges.append((start, prev + 1))
    return ranges


def span_texts(text, ranges):
    return [text[s:e] for s, e in ranges]


def add_gold_columns(df):
    df = df.copy()
    if "sample_id" not in df.columns:
        df["sample_id"] = [f"vihos_{i}" for i in range(len(df))]
    df["content"] = df["content"].astype(str)
    df["gold_indices"] = df.apply(parse_index_spans, axis=1)
    df["gold_ranges"] = df["gold_indices"].apply(indices_to_ranges)
    df["gold_span_text"] = [" | ".join(span_texts(t, r)) for t, r in zip(df["content"], df["gold_ranges"])]
    df["has_gold_span"] = df["gold_indices"].apply(lambda x: len(x) > 0)
    df["n_gold_chars"] = df["gold_indices"].apply(len)
    return df

train_df = add_gold_columns(train_df)
dev_df = add_gold_columns(dev_df)
test_df = add_gold_columns(test_df)

input_audit = pd.DataFrame([
    {
        "split": name,
        "n_rows": len(df),
        "n_with_span": int(df["has_gold_span"].sum()),
        "span_rate": float(df["has_gold_span"].mean()),
        "avg_gold_chars": float(df["n_gold_chars"].mean()),
        "median_gold_chars": float(df["n_gold_chars"].median()),
    }
    for name, df in [("train", train_df), ("dev", dev_df), ("test", test_df)]
])

input_audit.to_csv(RESULTS_DIR / "vihos_span_input_audit.csv", index=False, encoding="utf-8-sig")
input_audit

,split,n_rows,n_with_span,span_rate,avg_gold_chars,median_gold_chars
0,train,8844,4292,0.485301,8.752827,0.0
1,dev,1106,537,0.485533,9.101266,0.0
2,test,1106,531,0.480108,8.373418,0.0


## 5. Rule baseline tối ưu

In [5]:
def basic_norm(text):
    text = str(text).lower()
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def compact_norm_with_map(text):
    norm_chars = []
    raw_map = []
    last = ""
    repeat_count = 0
    for i, ch in enumerate(str(text)):
        c = unicodedata.normalize("NFC", ch.lower())
        if not (c.isalnum() or c == "đ"):
            continue
        if c == last:
            repeat_count += 1
            if repeat_count > 1:
                continue
        else:
            last = c
            repeat_count = 0
        norm_chars.append(c)
        raw_map.append(i)
    return "".join(norm_chars), raw_map


def compact_norm(text):
    return compact_norm_with_map(text)[0]


def get_phrase_column(df):
    for col in ["phrase_norm", "phrase", "span_text", "toxic_phrase"]:
        if col in df.columns:
            return col
    raise ValueError("Khong tim thay cot phrase trong toxic_phrases_candidates_train.csv")

phrase_col = get_phrase_column(phrases_df)
rules = phrases_df.copy()
rules["phrase"] = rules[phrase_col].astype(str).map(basic_norm)
rules["phrase_compact"] = rules["phrase"].map(compact_norm)
if "count" not in rules.columns:
    rules["count"] = 1
rules["count"] = pd.to_numeric(rules["count"], errors="coerce").fillna(1).astype(int)
rules["phrase_len"] = rules["phrase_compact"].str.len()

rules = rules[(rules["phrase_len"] >= 2) & (rules["phrase_compact"].str.contains(r"[a-zA-Z0-9à-ỹđ]", regex=True))].copy()

rules = rules[
    ((rules["phrase_len"] <= 2) & (rules["count"] >= 3)) |
    ((rules["phrase_len"] == 3) & (rules["count"] >= 2)) |
    ((rules["phrase_len"] >= 4) & (rules["count"] >= 1))
].copy()

rules = rules.sort_values(["phrase_len", "count"], ascending=[False, False])
rules = rules.drop_duplicates("phrase_compact").reset_index(drop=True)

if rules.empty:
    raise ValueError("Khong con rule nao sau khi loc. Kiem tra toxic phrases tu Note 6.")

rules[["phrase", "phrase_compact", "count", "phrase_len"]].head(20)

,phrase,phrase_compact,count,phrase_len
0,"học sinh nữ lột quần áo đánh nhau ngoài đường,...",họcsinhnữlộtquầnáođánhnhaungoàiđườnghọcsinhnam...,1,165
1,vào trường lớp giáo viên dạy ta bải vệ môi trư...,vàotrườnglớpgiáoviêndạytabảivệmôitrườngcâycốin...,1,158
2,các lãnh đạo tài giỏi và tin cậy của nhân dân ...,cáclãnhđạotàigiỏivàtincậycủanhândâncócáchsửlýô...,1,152
3,"trốn truy nã tại địa phương 26 năm, làm chánh ...",trốntruynãtạiđịaphương26nămlàmchánhvănphòngtại...,1,97
4,lẽ ra họ chỉ xây cầu bằng xốp thôi.... nhưng v...,lẽrahọchỉxâycầubằngxốpthôinhưngvìlươngtâmcủang...,1,88
5,cảm ơn các vị lãnh đạo đảng ta đã tạo điều kiệ...,cảmơncácvịlãnhđạođảngtađãtạođiềukiệnchonhữngđứ...,1,88
6,"""xuyên tạc đường nối chính sách của đãng và nh...",xuyêntạcđườngnốichínhsáchcủađãngvànhàlướcbôinh...,1,79
7,chả ảnh hưởng đến người nghèo nên chăng anh tư...,chảảnhhưởngđếnngườinghèonênchănganhtưvấnchothủ...,1,75
8,biển nhiễm độc! giờ trâu bò lợn gà lũ cuốn luô...,biểnnhiễmđộcgiờtrâubòlợngàlũcuốnluônmiềntrungt...,1,74
9,xe chở súc vật mong người dân hãy tránh xa (đề...,xechởsúcvậtmongngườidânhãytránhxađềphòngsúcvật...,1,73


In [6]:
def build_trie(phrases):
    root = {}
    END = "__END__"
    for phrase in phrases:
        node = root
        for ch in phrase:
            node = node.setdefault(ch, {})
        node[END] = phrase
    return root


def trie_matches(norm_text, raw_map, trie):
    END = "__END__"
    matches = []
    n = len(norm_text)
    for i in range(n):
        node = trie
        best = None
        j = i
        while j < n and norm_text[j] in node:
            node = node[norm_text[j]]
            if END in node:
                best = (i, j + 1, node[END])
            j += 1
        if best is not None:
            s, e, phrase = best
            raw_indices = sorted(set(raw_map[k] for k in range(s, e) if k < len(raw_map)))
            if raw_indices:
                matches.append({
                    "start": min(raw_indices),
                    "end": max(raw_indices) + 1,
                    "phrase": phrase,
                    "raw_indices": raw_indices,
                })
    return matches


def remove_overlaps(matches):
    if not matches:
        return []
    matches = sorted(matches, key=lambda m: (-(m["end"] - m["start"]), m["start"]))
    selected = []
    used = set()
    for m in matches:
        idx = set(m["raw_indices"])
        if idx & used:
            continue
        selected.append(m)
        used.update(idx)
    return sorted(selected, key=lambda m: m["start"])


def predict_rule(text, trie):
    norm_text, raw_map = compact_norm_with_map(text)
    matches = remove_overlaps(trie_matches(norm_text, raw_map, trie))
    pred_indices = sorted(set(i for m in matches for i in m["raw_indices"]))
    pred_ranges = indices_to_ranges(pred_indices)
    return {
        "pred_indices": pred_indices,
        "pred_ranges": pred_ranges,
        "pred_span_text": " | ".join(span_texts(str(text), pred_ranges)),
        "matched_phrases": " | ".join(m["phrase"] for m in matches),
    }

rule_trie = build_trie(rules["phrase_compact"].tolist())
print("n_rules:", len(rules))

n_rules: 4031


## 6. Hàm đánh giá span

In [7]:
def binary_scores(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    return float(p), float(r), float(f1)


def evaluate_predictions(df, method_name, split_name):
    total_tp = total_fp = total_fn = 0
    y_true_comment = []
    y_pred_comment = []

    for _, row in df.iterrows():
        gold = set(row["gold_indices"])
        pred = set(row["pred_indices"])
        total_tp += len(gold & pred)
        total_fp += len(pred - gold)
        total_fn += len(gold - pred)
        y_true_comment.append(int(len(gold) > 0))
        y_pred_comment.append(int(len(pred) > 0))

    char_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    char_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    char_f1 = (2 * char_precision * char_recall / (char_precision + char_recall)) if (char_precision + char_recall) else 0.0
    comment_precision, comment_recall, comment_f1 = binary_scores(y_true_comment, y_pred_comment)

    return {
        "method": method_name,
        "split": split_name,
        "n_rows": len(df),
        "n_gold_positive": int(sum(y_true_comment)),
        "n_pred_positive": int(sum(y_pred_comment)),
        "comment_precision": comment_precision,
        "comment_recall": comment_recall,
        "comment_f1": comment_f1,
        "char_precision": float(char_precision),
        "char_recall": float(char_recall),
        "char_f1": float(char_f1),
        "tp_chars": int(total_tp),
        "fp_chars": int(total_fp),
        "fn_chars": int(total_fn),
    }


def add_error_type(df):
    df = df.copy()
    err = []
    for _, row in df.iterrows():
        gold = set(row["gold_indices"])
        pred = set(row["pred_indices"])
        if not gold and pred:
            err.append("false_positive")
        elif gold and not pred:
            err.append("false_negative")
        elif gold and pred and gold != pred:
            err.append("partial_match")
        else:
            err.append("correct")
    df["error_type"] = err
    return df

## 7. Chạy rule baseline trên full dev/test

In [8]:
def run_rule_split(df, split_name):
    t0 = time.time()
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"rule {split_name}"):
        pred = predict_rule(row["content"], rule_trie)
        rows.append(pred)
    pred_cols = pd.DataFrame(rows)
    out = pd.concat([df.reset_index(drop=True), pred_cols], axis=1)
    out = add_error_type(out)
    metrics = evaluate_predictions(out, "rule_compact_longest", split_name)
    metrics["runtime_sec"] = time.time() - t0
    metrics["rows_per_sec"] = len(df) / metrics["runtime_sec"] if metrics["runtime_sec"] > 0 else np.nan
    return metrics, out

rule_metrics = []
rule_predictions = []
for split_name, split_df in [("dev", dev_df), ("test", test_df)]:
    metrics, pred_df = run_rule_split(split_df, split_name)
    rule_metrics.append(metrics)
    rule_predictions.append(pred_df.assign(method="rule_compact_longest", split=split_name))

rule_metrics_df = pd.DataFrame(rule_metrics)
rule_pred_df = pd.concat(rule_predictions, ignore_index=True)
rule_metrics_df

rule dev:   0%|          | 0/1106 [00:00<?, ?it/s]

rule test:   0%|          | 0/1106 [00:00<?, ?it/s]

,method,split,n_rows,n_gold_positive,n_pred_positive,comment_precision,comment_recall,comment_f1,char_precision,char_recall,char_f1,tp_chars,fp_chars,fn_chars,runtime_sec,rows_per_sec
0,rule_compact_longest,dev,1106,537,882,0.592971,0.973929,0.737139,0.477525,0.504470,0.490628,5078,5556,4988,0.296391,3731.563175
1,rule_compact_longest,test,1106,531,882,0.582766,0.967985,0.727530,0.481598,0.529856,0.504576,4907,5282,4354,0.282129,3920.188538


## 8. Dataset cho token classification

In [9]:
LABEL2ID = {"O": 0, "B-TOXIC": 1, "I-TOXIC": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}


def span_starts(indices):
    idx = set(indices)
    return {i for i in idx if (i - 1) not in idx}


class VihosTokenDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.df = df.reset_index(drop=True).copy()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["content"])
        gold = set(row["gold_indices"])
        starts = span_starts(gold)

        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_offsets_mapping=True,
        )

        labels = []
        for s, e in enc["offset_mapping"]:
            if s == e:
                labels.append(-100)
                continue
            inter = sorted(i for i in range(s, e) if i in gold)
            if not inter:
                labels.append(LABEL2ID["O"])
            else:
                labels.append(LABEL2ID["B-TOXIC"] if inter[0] in starts else LABEL2ID["I-TOXIC"])

        item = {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "offset_mapping": torch.tensor(enc["offset_mapping"], dtype=torch.long),
            "sample_idx": torch.tensor(idx, dtype=torch.long),
        }
        if "token_type_ids" in enc:
            item["token_type_ids"] = torch.tensor(enc["token_type_ids"], dtype=torch.long)
        return item


def move_batch_to_device(batch, device):
    keep_cpu = {"offset_mapping", "sample_idx"}
    return {k: (v if k in keep_cpu else v.to(device)) for k, v in batch.items()}

## 9. Train transformer token classifier

In [10]:
def token_predictions_to_df(model, tokenizer, df, split_name, batch_size=BATCH_SIZE):
    dataset = VihosTokenDataset(df, tokenizer, MAX_LENGTH)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    model.eval()

    rows = []
    token_true = []
    token_pred = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"predict {split_name}"):
            batch_dev = move_batch_to_device(batch, DEVICE)
            outputs = model(
                input_ids=batch_dev["input_ids"],
                attention_mask=batch_dev["attention_mask"],
                labels=batch_dev["labels"],
            )
            pred_ids = outputs.logits.argmax(dim=-1).cpu().numpy()
            true_ids = batch["labels"].numpy()
            offsets = batch["offset_mapping"].numpy()
            sample_indices = batch["sample_idx"].numpy()

            for b, sample_idx in enumerate(sample_indices):
                row = df.iloc[int(sample_idx)]
                text = str(row["content"])
                pred_chars = set()
                for j, pred_label in enumerate(pred_ids[b]):
                    true_label = int(true_ids[b][j])
                    if true_label != -100:
                        token_true.append(int(true_label != LABEL2ID["O"]))
                        token_pred.append(int(pred_label != LABEL2ID["O"]))
                    if pred_label == LABEL2ID["O"]:
                        continue
                    s, e = offsets[b][j]
                    if s == e:
                        continue
                    for pos in range(int(s), min(int(e), len(text))):
                        pred_chars.add(pos)

                pred_indices = sorted(pred_chars)
                pred_ranges = indices_to_ranges(pred_indices)
                rows.append({
                    "row_id": int(sample_idx),
                    "sample_id": row.get("sample_id", int(sample_idx)),
                    "content": text,
                    "gold_indices": row["gold_indices"],
                    "gold_ranges": row["gold_ranges"],
                    "gold_span_text": row["gold_span_text"],
                    "pred_indices": pred_indices,
                    "pred_ranges": pred_ranges,
                    "pred_span_text": " | ".join(span_texts(text, pred_ranges)),
                    "matched_phrases": "",
                })

    pred_df = pd.DataFrame(rows)
    pred_df = add_error_type(pred_df)
    metrics = evaluate_predictions(pred_df, "xlmr_token_classifier", split_name)
    p, r, f1 = binary_scores(token_true, token_pred)
    metrics.update({
        "token_precision": p,
        "token_recall": r,
        "token_f1": f1,
    })
    return metrics, pred_df


def train_token_classifier():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if not tokenizer.is_fast:
        raise ValueError("Tokenizer can phai la fast tokenizer de lay offset_mapping.")

    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL2ID),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    ).to(DEVICE)

    train_dataset = VihosTokenDataset(train_df, tokenizer, MAX_LENGTH)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    best_f1 = -1.0
    best_epoch = -1
    bad_epochs = 0
    logs = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        t0 = time.time()
        running_loss = 0.0

        for batch in tqdm(train_loader, desc=f"train epoch {epoch}"):
            batch = move_batch_to_device(batch, DEVICE)
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    labels=batch["labels"],
                )
                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += float(loss.detach().cpu())

        train_loss = running_loss / max(1, len(train_loader))
        dev_metrics, _ = token_predictions_to_df(model, tokenizer, dev_df, "dev", batch_size=BATCH_SIZE)
        epoch_time = time.time() - t0

        log = {
            "epoch": epoch,
            "train_loss": train_loss,
            "dev_char_f1": dev_metrics["char_f1"],
            "dev_char_precision": dev_metrics["char_precision"],
            "dev_char_recall": dev_metrics["char_recall"],
            "dev_comment_f1": dev_metrics["comment_f1"],
            "runtime_sec": epoch_time,
        }
        logs.append(log)
        print(log)

        if dev_metrics["char_f1"] > best_f1:
            best_f1 = dev_metrics["char_f1"]
            best_epoch = epoch
            bad_epochs = 0
            TOKEN_MODEL_DIR.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(TOKEN_MODEL_DIR)
            tokenizer.save_pretrained(TOKEN_MODEL_DIR)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("Early stopping")
                break

    log_df = pd.DataFrame(logs)
    log_df["best_epoch"] = best_epoch
    log_df.to_csv(RESULTS_DIR / "vihos_token_model_training_log.csv", index=False, encoding="utf-8-sig")

    best_model = AutoModelForTokenClassification.from_pretrained(TOKEN_MODEL_DIR).to(DEVICE)
    best_tokenizer = AutoTokenizer.from_pretrained(TOKEN_MODEL_DIR, use_fast=True)
    return best_model, best_tokenizer, log_df

In [11]:
if RUN_TOKEN_MODEL:
    token_model, token_tokenizer, train_log_df = train_token_classifier()
else:
    token_model = None
    token_tokenizer = None
    train_log_df = pd.DataFrame()
    print("RUN_TOKEN_MODEL = False, bo qua train token classifier.")

train_log_df.tail()

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

d:\Thư bae ngốc ngếch\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SS\.cache\huggingface\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\SS\AppData\Local\Temp\ipykernel_12536\1203463226.py:87: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` in

train epoch 1:   0%|          | 0/1106 [00:00<?, ?it/s]

C:\Users\SS\AppData\Local\Temp\ipykernel_12536\1203463226.py:102: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):


KeyboardInterrupt: 

## 10. Đánh giá token model trên dev/test

In [ ]:
token_metrics_df = pd.DataFrame()
token_pred_df = pd.DataFrame()

if RUN_TOKEN_MODEL:
    token_metrics = []
    token_predictions = []
    for split_name, split_df in [("dev", dev_df), ("test", test_df)]:
        t0 = time.time()
        metrics, pred_df = token_predictions_to_df(token_model, token_tokenizer, split_df, split_name, batch_size=BATCH_SIZE)
        metrics["runtime_sec"] = time.time() - t0
        metrics["rows_per_sec"] = len(split_df) / metrics["runtime_sec"] if metrics["runtime_sec"] > 0 else np.nan
        token_metrics.append(metrics)
        token_predictions.append(pred_df.assign(method="xlmr_token_classifier", split=split_name))

    token_metrics_df = pd.DataFrame(token_metrics)
    token_pred_df = pd.concat(token_predictions, ignore_index=True)

token_metrics_df

## 11. So sánh rule và model

In [ ]:
metrics_df = pd.concat([rule_metrics_df, token_metrics_df], ignore_index=True)
metrics_df = metrics_df.sort_values(["split", "char_f1"], ascending=[True, False]).reset_index(drop=True)
metrics_df.to_csv(RESULTS_DIR / "vihos_span_baseline_metrics.csv", index=False, encoding="utf-8-sig")

predictions_df = pd.concat([rule_pred_df, token_pred_df], ignore_index=True)
predictions_export = predictions_df.copy()
for col in ["gold_indices", "gold_ranges", "pred_indices", "pred_ranges"]:
    if col in predictions_export.columns:
        predictions_export[col] = predictions_export[col].map(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, tuple, set)) else x)
predictions_export.to_csv(RESULTS_DIR / "vihos_span_baseline_predictions.csv", index=False, encoding="utf-8-sig")

metrics_df

In [ ]:
dev_scores = metrics_df[metrics_df["split"] == "dev"].sort_values("char_f1", ascending=False)
best_method = dev_scores.iloc[0]["method"]

selection_df = dev_scores[[
    "method", "split", "char_precision", "char_recall", "char_f1",
    "comment_precision", "comment_recall", "comment_f1", "runtime_sec", "rows_per_sec"
]].copy()
selection_df["selected"] = selection_df["method"] == best_method
selection_df.to_csv(RESULTS_DIR / "vihos_span_setting_selection.csv", index=False, encoding="utf-8-sig")

print("Best method:", best_method)
selection_df

## 12. Phân tích lỗi

In [ ]:
selected_errors = predictions_df[
    (predictions_df["method"] == best_method) &
    (predictions_df["split"].isin(["dev", "test"])) &
    (predictions_df["error_type"] != "correct")
].copy()

error_export = selected_errors.copy()
for col in ["gold_indices", "gold_ranges", "pred_indices", "pred_ranges"]:
    if col in error_export.columns:
        error_export[col] = error_export[col].map(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, tuple, set)) else x)

error_export.to_csv(RESULTS_DIR / "vihos_span_baseline_errors.csv", index=False, encoding="utf-8-sig")

error_summary = (
    selected_errors.groupby(["split", "error_type"])
    .size()
    .reset_index(name="n")
    .sort_values(["split", "n"], ascending=[True, False])
)
error_summary.to_csv(RESULTS_DIR / "vihos_span_error_summary.csv", index=False, encoding="utf-8-sig")
error_summary

In [ ]:
cols = ["split", "error_type", "content", "gold_span_text", "pred_span_text", "matched_phrases"]
for err_type in ["false_positive", "false_negative", "partial_match"]:
    print("\n", "=" * 20, err_type, "=" * 20)
    display(selected_errors[selected_errors["error_type"] == err_type][cols].head(5))

## 13. Audit rule phrase

In [ ]:
rule_audit = rules[["phrase", "phrase_compact", "count", "phrase_len"]].copy()

if not rule_pred_df.empty:
    phrase_hits = []
    for s in rule_pred_df["matched_phrases"].fillna(""):
        for p in str(s).split(" | "):
            p = p.strip()
            if p:
                phrase_hits.append(p)
    hit_df = pd.Series(phrase_hits, name="phrase_compact").value_counts().reset_index()
    hit_df.columns = ["phrase_compact", "n_pred_hits"]
    rule_audit = rule_audit.merge(hit_df, on="phrase_compact", how="left")
else:
    rule_audit["n_pred_hits"] = 0

rule_audit["n_pred_hits"] = rule_audit["n_pred_hits"].fillna(0).astype(int)
rule_audit.to_csv(RESULTS_DIR / "vihos_span_rule_audit.csv", index=False, encoding="utf-8-sig")
rule_audit.head(20)

## 14. Runtime benchmark

In [ ]:
runtime_cols = ["method", "split", "n_rows", "runtime_sec", "rows_per_sec"]
runtime_df = metrics_df[[c for c in runtime_cols if c in metrics_df.columns]].copy()
runtime_df.to_csv(RESULTS_DIR / "vihos_span_runtime_benchmark.csv", index=False, encoding="utf-8-sig")
runtime_df

## 15. Export artifact cho app

In [ ]:
rules_payload = {
    "version": "note7_rule_baseline_v1",
    "normalization": "compact_norm_keep_alnum_collapse_repeats",
    "n_rules": int(len(rules)),
    "rules": rules[["phrase", "phrase_compact", "count", "phrase_len"]].to_dict(orient="records"),
}

RULES_JSON = RESOURCES_DIR / "toxic_span_highlighter_rules.json"
with open(RULES_JSON, "w", encoding="utf-8") as f:
    json.dump(rules_payload, f, ensure_ascii=False, indent=2)

explainer_config = {
    "version": "note7_span_explainer_v1",
    "selected_method": best_method,
    "rule_path": str(RULES_JSON.relative_to(PROJECT_ROOT)),
    "token_model_path": str(TOKEN_MODEL_DIR.relative_to(PROJECT_ROOT)) if TOKEN_MODEL_DIR.exists() else None,
    "model_name": MODEL_NAME if RUN_TOKEN_MODEL else None,
    "label2id": LABEL2ID,
    "id2label": ID2LABEL,
    "max_length": MAX_LENGTH,
    "fallback_method": "rule_compact_longest",
}

CONFIG_JSON = RESOURCES_DIR / "span_explainer_config.json"
with open(CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(explainer_config, f, ensure_ascii=False, indent=2)

print("Saved:")
print(RULES_JSON)
print(CONFIG_JSON)
if TOKEN_MODEL_DIR.exists():
    print(TOKEN_MODEL_DIR)

## 16. Tổng kết

In [ ]:
summary = {
    "project_root": str(PROJECT_ROOT),
    "best_method": best_method,
    "metrics_path": str((RESULTS_DIR / "vihos_span_baseline_metrics.csv").relative_to(PROJECT_ROOT)),
    "predictions_path": str((RESULTS_DIR / "vihos_span_baseline_predictions.csv").relative_to(PROJECT_ROOT)),
    "errors_path": str((RESULTS_DIR / "vihos_span_baseline_errors.csv").relative_to(PROJECT_ROOT)),
    "rules_path": str(RULES_JSON.relative_to(PROJECT_ROOT)),
    "config_path": str(CONFIG_JSON.relative_to(PROJECT_ROOT)),
    "token_model_path": str(TOKEN_MODEL_DIR.relative_to(PROJECT_ROOT)) if TOKEN_MODEL_DIR.exists() else None,
}
summary